# Session 1: a magnet and a coil make a voltage
**A. Real-world problem.** A bicycle dynamo has no battery, yet it lights a lamp when you pedal.
Inside is a magnet and a coil of wire. Nothing touches, nothing is charged up. Where does the
voltage come from? The answer is *induction*: a **changing** magnetic field through a coil
pushes charges round the wire.

**Plan.** (1) push a magnet through a coil and measure the voltage, (2) see that speed and
turns matter, (3) spin a coil and get an alternating voltage.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

AREA = 0.002          # coil area in m^2 (about 4.5 cm x 4.5 cm)

## Step 1: the one rule
*Voltage = number of turns x how fast the flux changes.*  Flux (symbol Phi, unit weber, Wb)
is the amount of magnetic field passing through the coil. If the flux changes by `delta_flux`
in a time `delta_t`, a coil of `turns` turns gives an average voltage

    emf = turns * delta_flux / delta_t      (volts)

Check by hand: 200 turns, flux change 0.0005 Wb in 0.05 s gives 200 * 0.0005 / 0.05 = 2 V.
Twice as fast (0.025 s) gives 4 V. No change of flux, no voltage, however strong the magnet.

In [ ]:
def average_emf(turns, delta_flux, delta_t):
    """Size of the average voltage (V) while the flux changes by delta_flux (Wb) in delta_t (s)."""
    if delta_t <= 0:
        raise ValueError("delta_t must be positive")
    return abs(turns * delta_flux / delta_t)

## Step 2: a magnet pushed through a coil
We model the flux through the coil as a bell-shaped bump: it is largest when the magnet is
in the middle of the coil and small when the magnet is far away (a simple, illustrative
shape, not a measurement). The voltage is minus turns times the slope of that curve.
The minus sign is Lenz's law: the induced voltage opposes the change. Going in, the voltage
has one sign; coming out, the opposite sign.

In [ ]:
PHI_MAX = 0.001       # flux with the magnet in the middle, Wb (illustrative)
WIDTH = 0.04          # how far (m) the magnet must be from the middle for the flux to halve


def magnet_flux(x):
    """Flux (Wb) through the coil with the magnet at position x (m) from the coil middle."""
    return PHI_MAX / (1.0 + (np.asarray(x, dtype=float) / WIDTH) ** 2)


def magnet_pass(speed, turns, span=0.4, n=4001):
    """Push the magnet through at constant speed (m/s). Returns time (s), flux (Wb), emf (V)."""
    t = np.linspace(0.0, span / speed, n)
    x = -span / 2 + speed * t
    phi = magnet_flux(x)
    emf = -turns * np.gradient(phi, t)
    return t, phi, emf

## Step 3: a spinning coil makes alternating voltage
Turn a coil of area A in a field B. When the coil faces the field the flux is B*A; when it is
edge-on the flux is zero. Spinning keeps changing the flux, so the voltage keeps changing,
and it swings positive and negative: **alternating current (AC)**. The peak voltage is

    peak = turns * B * A * 2 pi f      (f = turns per second = frequency in hertz)

Check by hand: 100 turns, B = 0.1 T, A = 0.002 m^2, f = 5 Hz gives
100 * 0.1 * 0.002 * 2 pi * 5 = 0.628 V.

In [ ]:
def peak_emf(turns, b_field, area, freq):
    """Peak voltage (V) of a coil of `turns` turns, area (m^2) spinning at freq (Hz) in field b_field (T)."""
    return turns * b_field * area * 2 * np.pi * freq


def rotating_emf(t, turns, b_field, area, freq):
    """Voltage (V) at times t (s). Starts at zero when the coil faces the field."""
    return peak_emf(turns, b_field, area, freq) * np.sin(2 * np.pi * freq * np.asarray(t, dtype=float))

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    for speed, col in ((0.5, "tab:blue"), (1.0, "tab:red")):
        t, phi, emf = magnet_pass(speed, 200)
        axes[0].plot(t * 1000, emf, color=col, label=f"magnet speed {speed} m/s")
        print(f"speed {speed} m/s: peak voltage {abs(emf).max():.2f} V, passes in {t[-1] * 1000:.0f} ms")
    axes[0].set_xlabel("time (ms)")
    axes[0].set_ylabel("coil voltage (V), 200 turns")
    axes[0].set_title("Faster magnet: bigger, shorter pulse")
    axes[0].grid(alpha=0.3)
    axes[0].legend(fontsize=8)
    tt = np.linspace(0, 0.4, 800)
    for n, col in ((50, "tab:green"), (100, "tab:orange"), (200, "tab:purple")):
        axes[1].plot(tt * 1000, rotating_emf(tt, n, 0.1, AREA, 5), color=col, label=f"{n} turns")
    axes[1].set_xlabel("time (ms)")
    axes[1].set_ylabel("coil voltage (V)")
    axes[1].set_title("Spinning coil (5 Hz, 0.1 T): alternating voltage")
    axes[1].grid(alpha=0.3)
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "magnet_and_coil.png", dpi=120)
    print("average emf 200 turns, 0.5 mWb in 0.05 s:", average_emf(200, 0.0005, 0.05), "V")
    print("peak emf 100 turns, 0.1 T, 5 Hz:", round(peak_emf(100, 0.1, AREA, 5), 3), "V")

## Where this stops
Here we only saw *that* the voltage depends on how fast the flux changes. Session 2 turns
this into Faraday's law with units, gives the exact sine wave, and adds the transformer.

## Coding tasks
1. **Easy.** In `lesson.py` change the magnet speed from 0.5 and 1.0 to 0.25 and 2.0 m/s. Predict the two peak voltages first, then run it and compare.
2. **Stretch.** Using `peak_emf`, find the frequency (in whole hertz) needed for a peak of 5 V with 200 turns, 0.15 T and 0.002 m^2, and check it against the app challenge.
3. **Explain.** Explain, in your own words, why the voltage from a magnet pushed through a coil has one sign going in and the opposite sign coming out.

## Assessment
- **A. Conceptual.** Why does a strong magnet resting inside a coil make no voltage? And where does the electrical energy come from when you push the magnet in?
- **B. Numerical.** A coil of 500 turns sees the flux through it grow from 0 to 0.2 mWb in 0.02 s. What is the average voltage? Check that the answer is sensible.
- **C. Graph and simulation interpretation.** In the left figure the magnet speed is doubled. What happens to (i) the peak voltage, (ii) how long the pulse lasts, (iii) the area of one pulse?
- **D. Design and optimisation challenge.** Design: you want a spinning coil with a peak voltage of 3 V. You have N = 100 turns, A = 0.002 m^2 and a 0.1 T magnet. What frequency do you need? If your hand can only turn the coil at 5 Hz, how many turns would you need instead? What is the trade-off?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: 200 turns, 0.5 mWb in 0.05 s gives 2 V; twice as fast gives 4 V",
                   abs(L.average_emf(200, 0.0005, 0.05) - 2.0) < 1e-12 and abs(L.average_emf(200, 0.0005, 0.025) - 4.0) < 1e-12))
    checks.append(("hand calculation: 100 turns, 0.1 T, 0.002 m^2, 5 Hz peak is 0.628 V",
                   abs(L.peak_emf(100, 0.1, 0.002, 5) - 0.6283185) < 1e-6))
    checks.append(("no change of flux gives no voltage", L.average_emf(500, 0.0, 1.0) == 0.0))
    checks.append(("double the turns doubles the average voltage",
                   abs(L.average_emf(400, 0.001, 0.1) / L.average_emf(200, 0.001, 0.1) - 2) < 1e-12))

    t1, _, e1 = L.magnet_pass(0.5, 200)
    t2, _, e2 = L.magnet_pass(1.0, 200)
    t3, _, e3 = L.magnet_pass(0.5, 400)
    checks.append(("twice the magnet speed doubles the peak voltage", abs(abs(e2).max() / abs(e1).max() - 2) < 0.01))
    checks.append(("twice the turns doubles the peak voltage", abs(abs(e3).max() / abs(e1).max() - 2) < 0.01))
    area1 = np.trapezoid(np.clip(e1, 0, None), t1)
    area2 = np.trapezoid(np.clip(e2, 0, None), t2)
    checks.append(("the area of one pulse (volt-seconds) does not depend on speed", abs(area1 - area2) < 1e-4))
    checks.append(("Lenz: voltage going in and coming out have opposite signs",
                   e1[:len(e1) // 2].mean() * e1[len(e1) // 2:].mean() < 0))

    t = np.linspace(0, 0.4, 4001)
    e = L.rotating_emf(t, 100, 0.1, 0.002, 5)
    checks.append(("a rotating coil gives an alternating voltage (both signs)", e.max() > 0.6 and e.min() < -0.6))
    checks.append(("the voltage never exceeds the peak formula", abs(e).max() <= L.peak_emf(100, 0.1, 0.002, 5) + 1e-12))
    checks.append(("voltage is zero at time zero (coil facing the field)", abs(L.rotating_emf(0.0, 100, 0.1, 0.002, 5)) < 1e-12))
    try:
        L.average_emf(100, 0.001, 0.0)
        ok = False
    except ValueError:
        ok = True
    checks.append(("a zero time interval is rejected", ok))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
